# Pipeline dan evaluasi pengelompokan perangkat elektronik

Notebook ini memandu pembaca dari input hingga interpretasi hasil. Sel perhitungan menggunakan artefak yang sudah disertakan. Ekstraksi *embedding*, eksperimen 100 seed, inferensi DMS46, dan eksperimen lain tidak dijalankan ulang. Baca [paper](../paper/KARYA-ILMIAH-SD2026040000061-4-CONTOH.pdf) untuk metode lengkap dan batas klaimnya.

## 1. Citra dan penyeragaman input

Data internal berasal dari foto dengan riwayat akuisisi berbeda. Pemeriksaan awal dan penyeragaman ukuran serta JPEG dijelaskan pada [laporan akuisisi](../reports/02-HARMONISASI.md). Implementasi ada pada [`harmonize.py`](../scripts/harmonize.py) dan [`embed_harmonized.py`](../scripts/embed_harmonized.py). Notebook ini tidak memerlukan citra mentah.

![Alur pengelompokan](../paper/figures/fig1_architecture.png)

## 2. Embedding, fusion, dan clustering

Delapan *frozen backbone* menghasilkan representasi visual. Runner [`run_semantic_g0.py`](../scripts/run_semantic_g0.py) memuat konfigurasi fusion dan K-Means. Ruang Fusion pada pemilihan K dan Fusion-L2 pada transfer adalah dua konfigurasi berbeda; lihat [konteks versi](../LATEST-PAPER-CONTEXT.md).

In [ ]:
from pathlib import Path
import sys

root = Path.cwd()
if not (root / 'tables').is_dir():
    root = root.parent
assert (root / 'tables').is_dir(), 'Jalankan notebook dari folder notebooks/ atau root repo'
sys.path.insert(0, str(root))
from reporting.evidence import EvidenceRepository
evidence = EvidenceRepository(root)


## 3. Pemilihan K dan stabilitas

K=12 dipilih dalam rentang lanjutan K=12–16. Ini bukan bukti jumlah cluster yang benar secara universal. Tabel berikut berasal dari 100 seed dan chooseR. Runner dan hasil lengkap tersedia pada [`run_k_stability_100.py`](../scripts/run_k_stability_100.py) dan [`summary.json`](../tables/k_stability_100/summary.json).

In [ ]:
from IPython.display import Markdown, display

rows = evidence.csv('k_stability_100/chooser_summary.csv')
columns = ['k', 'chooseR_winner']
for candidate in ('consensus_silhouette', 'source_nmi'):
    if candidate in rows[0]:
        columns.append(candidate)
display(Markdown('| ' + ' | '.join(columns) + ' |\n|' + '|'.join(['---'] * len(columns)) + '|\n' + '\n'.join('| ' + ' | '.join(row[col] for col in columns) + ' |' for row in rows)))
print('Pilihan K dan rata-rata ARI:', evidence.selected_k())


## 4. Transfer ke citra Bangladesh

*Transform*, centroid, dan pemetaan internal diterapkan pada 710 citra dari empat class Bangladesh. Sel berikut menghitung ulang macro Top-1 langsung dari prediksi per citra. Ini evaluasi label fungsi pada data external, bukan pelatihan ulang model. Lihat [`strict_transfer_predictions.csv`](../tables/final_claim_checks/strict_transfer_predictions.csv).

In [ ]:
transfer = evidence.strict_transfer()
for key, value in transfer.items():
    print(f'{key}: {value}')


![Contoh cluster](../paper/figures/fig4_atlas.png)

## 5. Profil material visual

Apple DMS46 memprediksi material yang tampak pada citra. Nilai holdout dan external di bawah dibaca dari tabel hasil, karena prediksi material per citra tidak disertakan di repo. Nilai tersebut tidak dihitung ulang oleh notebook. Lihat [`material_holdout_metrics.csv`](../tables/final_claim_checks/material_holdout_metrics.csv), [`roi_metrics.json`](../tables/bangladesh_material/roi_metrics.json), dan runner [`run_bangladesh_dms46_roi.py`](../scripts/run_bangladesh_dms46_roi.py).

In [ ]:
for key, value in evidence.material().items():
    print(f'{key}: {value}')


![Contoh profil material](../paper/figures/fig7_material_examples.png)

## 6. Membaca hasil

Hasil pengelompokan perangkat dan profil material visual memberi dua jenis informasi. K-Means Fusion K=12 stabil pada 100 seed, sedangkan evaluasi transfer Fusion-L2 pada empat class Bangladesh memberi macro Top-1 sekitar 0,888. Prediksi DMS46 pada ROI external memberi macro AUROC sekitar 0,732. Angka material hanya menggambarkan permukaan terlihat dan belum memvalidasi komposisi fisik atau keputusan pemilahan. [Evidence manager](../paper/EVIDENCE-MANAGER.md) menjelaskan klaim dan batasnya.